# Weak-Preference RSD with Pareto Completion and Nash-Welfare Evaluation


This CPU-ready notebook reuses cached LLM profiles. Weak thresholds remain selected only on validation PE/blocking outcomes; Nash welfare is evaluated on the locked test market.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Download the dataset ZIP from Google Drive.
FILE_ID = "1Pi_5JOhoCK4pELRhu0iv52us5DCwOJMd"
ZIP_PATH = Path("fixed_tolerance_course_data.zip")

# Download again if the local file is missing or is not a valid ZIP.
if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )

    assert downloaded_path is not None, (
        "The Google Drive download failed. Make sure the file is shared "
        "with 'Anyone with the link'."
    )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


In [ ]:
# Restore the previously completed 128-student LLM-profile checkpoint.
# The compressed checkpoint is embedded, so no GPU or LLM call is needed.
import base64, gzip
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"
if not CHECKPOINT.exists():
    CHECKPOINT.write_bytes(gzip.decompress(base64.b64decode("H4sIAAyM8moC/+1dW48dN3J+z68Y6HlOg1Us3vK2wC6CRYwNsLt5CFaG4FgKIERrGbogCSz/9zTZ0tF4m93NJots9tExsIvxjI4t22Sx6qvv8suz9x8+vnz104cXr18+++cH8fgwfuOHDx/fj3/y7O1/Pxv//Od3b//r9ZtX4zd+efbj24/v3r968f7Ht+9evQ/fEf5Tg/8cXL/C8SsIX8nr9+j6lbp+pa9fmetXdvzqMn3Yff1Li+tfET7/bX4dv371vz+/ef3j6w9v/m/6Hb18Mf3+/O/sb/j4IB8f6PFBPT7oxwf7+OAeH0B8P37ux7d///nNq7+P/9ThV/5Nhh/4n7z/+J/vP7z+8PHDq+tP7Pff+7/Xux/+58Xbjx9+/vjB/5v55flPDw/Pv/zr+OGD/8zLV++f+w/5Hz08/HL96euX/tvjb+f5sy+/0H/j+bPv/vivf3j+7NfH+AdA7P2EnX/g93/8y/pn5Pwzf/rDv//1z7/7bvkzlPEZlfEZnfEZt/YZ/5HvH7/8p7uegem/2jP5CcTzZ19+/uQkXH9uxx8//+lXfylevXv39t34/Z8+vnnjz82rH9/+9NKfGYABwRiyRio3/mH0r//0y28vGRRess/XA6/f9LcMI7fs8y/87TWD6zUTsVsm9t2y8ReP/zj/cNfM1+s2/g9iV85/Inrjxr+a4rlxkaP9l7/++d/+9C/f/ceL9Uth9t48kXFSIeO24nG3dfP3lnNbLfcN31FLtz8E+bUEP8m1UgKfVEopUYNz4EihBDH+QWpWSrCwlMQqCVwryef6sf1gb1USTKgkEH2vo8UDwg+aV4+N93rvBzTPZd68mKrniwnbRXr9psEnAIarBoOUwmqLBP7VVjC7ajLrqkGkNY492mxXDdIe7emqxa+XWLxeemqUj32dd/fFsPtqrl6z9fMoNs6jnrrMrQOpB2dQEoCyvvRrOTuPVGFUw9VRDfrrIU2/x1TvPXS7J8TdbWqz1ompH+5mErXN20fDUEN8/6g1OEOoXZhFZ0VEFRYRjDxqF9gN+HytIpBdRbJKyPghFy0hU23hqCCQXUFypj2Z0SD2jOeY45rX/fBc1ZoAn9xaTZhKxmZJoMEJZeX4f873FWbeV+isknCBgsYiVhLguMai25YCGRDh3UNslVIgGgFMplFTcQJQiqOdMAqNJhBmbCastLPaYSq0E7H1ERSNyIndxFQuFpdG/sex1uHgpkG1gKH4j/v626c+6eW3b/PgmsECOT9HG//myTm2Y6vCqGVrT9x9cOUXADV+cGEZ3+HreuvDp7HzlD+a5zSVqZAQbGFCab2bGac5o0CT8qcYIotF13x7jxWPcdY0F9/j1zrSm6dmd8fGNFclXoSCO5RDNmgF8ZgDtxfp49wa2yBlkNMClCMlMTxqEZoBH5lHVBzkLpA8ye0sByqQdzrlGJxpNKtySXXPF5trNEttEVTgFpUuMceaYKUxYtphugi4A9DFhHYBnhEtid1HSxgvI6bT+bxmG6HCredCWsdEU4ENMaBDC1ZgGBD1fE8CyIeKljymgoEYO23/dzJhD34t95NhRX4xlmWtmRpIW+GsFIG1peasLZCN8AYswsl2dGYsGDsf2bo++qA46Svd9FeWZ8T9Vvu4Yu64GpR1bhzmUC6VDj7az2U/yHPZIP5gG+KPXtRruKprupy7kLPWqo/eUMfDpMn4l9iMr9oYJdIbmhSX2OHqgRxqbYzWocOdr+5AVQCPo1TyJ6oUSOUClWm/FkfD5UJyIJG8gVJMMzAHd/dHORPnjhWLLgdQ3KAMjl27leTxEzFfFIKugKlCIutWsD6+05Zw8V643haEusaCcKv6MlCu1DCWXGOd1DqwJmh+qAybtOAC+wWBvDIe8aWh+y0wt6Y1EN2C9Dn9nGhRKr8Jsd2+yyo2WqVEDB0H4/uk8REg3ynZyHW1XezVODjTuQOYXHgfDr6u7iDdbhWM5M7KPoBkLdde/CTBhR3nLCsoLOXJzYuHq0uvXpcRAvciYbV6LCzjulwwIINEsMGU1g0Zu2fAZVkXt7VNLKXkoBLgSIswPsr5+h0F2/o9it7C6ubn0odskxaI05+nzu7W8RVnBeqVrC14pGdVmoIcfLh5I0HreNSELCTAtmAVCuloWiXPhxGEI4kJ7AUFv7DXzUoPsegdxFdAjvUOquIDdAadx4ZtTuKtEQOa8RG2YKR/hzHyDmPVER66p8b6X95hF26rd+GtJn51Z8SWMWJprRSk9OKKjEDSECZxO6f0YQ0S1gXZWVhthc7VC0COSKRhy17lwt0a5evEaucklTPasX0Q1k1/zCsHsVUOWF0CX5ronPes62DJculMxnt439A1F7sZltWc1Ci0lShCXx+ZhhXbJh0S6Rm4OgxjlJ5Rk1i9slE/kF+NDISj+qxIDoQ+H2es0hLYwzH3CrOB4HHq1sP4yFurFE1eSPNyopt7LFaGCRZM8c/psGirV6Fb8zHUt4oWMBkmenc0UmOH4Sbyz7wiGD7v/qpUvcb2aFV5obVW+K1c+3t24Df3MWVbvZXoragG7ZQGsDixf+aWHGiriiz2L+swu3hMZcOv6RJLxLT2v/XpI497snz0pj3ytls4GiG1IRuYJwrnJ89VnYyj9vWXmGH4Z4S8mGSeZxB1lLU989xZ3yr8wJ1XmU4o/c1KX0mciAUPTPkDnsoKDiR5NqsQEJmOpWjlFNy4F06bjU3P7lLYgoUC92Sp9lC6YXGKksPYL6ByMEHpZg6lS6hKkXnSMDS53DtQr4Xm9fA7rVrMt3S/n3VmSErdcaES6OSkQHZuzl2TxVZUqfGR0MigP/nJleEWnp+1Bi1e55MagVcRdakSKhkOYJQ15NDb3js7N+WR8kBYJ+qqiK09eRreygbuGfl604ruybrjxZI4bs18qP+OTvbfGZttpzWiXXjVqYYyLBrF1ZxUVuLc3qsojF8LJu8bnyOIaVv32yWO04osEChaYJtL1YdOC/m79jS5iTlH496TygvurJBSrKxIQyIHZXwAOwYZmbORTGjdKj+zyaMd5us9KZl8jNKGj7MssJlbNwRL4x2qQSvhSKEKhmCRHB9p6toEtO8F1XqQmqu8SOm89Wsfp+bKdwh2EOiU1HZKEoU5WU5avp4HWegubNr07OUgI80Asw+1bNG96BYs8m91zZeKO+BAUliSioLvr4i8Na6CDJYp2CKrg9kZGNejDB4YZPD3BO8+8QYqnEicJSvAmQUvWRK3F8Fr1x28azsVN2we7Y36urAYKTtvhmS9+jOEY4g5iE7QXK91idlzc4Bs+5VaywB6l1KtWkw2fZxC4zbeKGTaeeHgyIE2arqukcwJwkJSesljVcsNUU2d50K7qbp9rDh8y/lfN+Q2CNzqz1T5Q2XHg2/HoYuET+oGPVdjkDxFNG9lC3F7Q9Liu5P4N+Jflk46szzEEZDgiJQIKOuce0bUHL+BnETfqrKuRVVXVcuTxFqiOawxaiFH2Kg2tEogFqetQZtEFeCxPHEKUGkIug8V6clVIwBJZCSEQw6CZJewI7HUjrsegsGBoyEvu5dbLjyKIZpQD5acVgDkdYYC5yQL0qeANJNceJjjTm9lQ17FW57uC472RjyaJ4WLBmcRpZQ2aC4iew7TKq40dW/ZXHLhP+cOSEG+Uae/KtJI1bGL7xn0F2J9cZQefwwknJHjKL0ERdvmwa619qYyUdNcNS2jyPQmjXWPw/gwaNLgZOBRzv+buvY+zdhZUiN2aNiM1fWyTC4TPcVO0nHpSj2uKDn8pJVX7QhfQIJqB+ajpxKn2NMkjZ5yQkJSHS96QE5tO1PhclK/NOM5svJz3sicaKngQBhjx7K7qjl51xvwJkiG4YkUPzVicrPoBwt9wFuvSSHRjZUkgPTzUoLNhxVMHVagRXwZrESIHoZ86IZQfytpaM9dp2hVRYoKgubxJbcKlTAAE5cu0qnKVnZt6zyArVkX2/qS08QMOBkBwN04+6gKbNpqDXOIseuar1wgBiT5uUoJmihsVOSclKiIL9sA9+eaYRfh5Gu0RCa0tFXsQEN96MkSDvLhh2+Em5gcTi4IpLahKXFzAF6pqnNKRlNSxCV6WkqijKLOcHbNYYXHb8XOPrvkPZypLgc0vpXCChlMYPRcdKZ0K7MObGTWcT3kqy+lOptlW77grApT9WZdlxSPpxoMVhsSCDjZ5Mx39spUlXtucIB443QELzGw6m6/ujtq/XiTO//ngMKgeQoDjZOskqSNVQtb2z5StljsmJeLwt2EubTMqHORigscpXM65JQX2hEoJZwOO28dWVS5CjdRsBuAXbDWC40riyquHhlKoq2w15QA2Wgj1Eoc28TarERhhlurqsTH2UpyqOwUYIJzmFm3dyh6YtISJcJACSqURYKx3Q7OPatzVAsvGMOTLphTFNlZAkVOZZalHGi0ciwDGkIk7rxF0NAs+AB7TdQOn1vSr6uutk4VQTR5nHFT/iL/rg7cJ+fZELGrxDAVNA6csBgw+XlNwVbJpE38da7Gw3bH3umcNjrYqzNxfmddzP6yXiE/DtpGUkjinYthtezjvCPneb9dJ6l8jSr/OH/3eerD54kGQQIkKRVA7djcTM2xtCdzM/u2K6tPdksLr1qG+rWgZ9Mrw1Oey1+uymTevIa41cVYErdTW3BoBIbiEekQDogfu2C9LI7PQRwLvBTTrRecOai/jSwEtvInVLndmBlAEaFy0i14O2nN9qhd9neuT8Bg1lGNiltXPjnkscNbFbRVdLy3vYWeNVGmKAeLxgCQokDQitxuwxdVn3q7sRL/d6tl7Uim2NBZv6XAsQ8L/1bYrOAppunrHiahohMKADXaBU2ArmG0JPab8jxhZ0G1orCfuEk15QI5L14tDxwWe9Mq+yHbqCWB4+IQDqV5qpCBlWRVLCxaCQH2dvNK4r5Vc55vx5Zns1dNdecx0igxjqEq0BBmZ8mIRpTh9FcJW7aqi3jJ44MHVPRZnOL0ATTi23rJoAfT/QKnLv+XVaGZTWhWjbKoCZ3yA+xcYGSg+QtzgUrUpKfpxeciEeTA/lmQCvZxjBNnLTM4ENqgBRtcouY0GIOniPbFZPX1Ip1WLqEshwP+UCP8ZWuUl+ujfFrHZK1xUumwjxZuvo82srnKan2hBCy10e3lZ9rT2Y256m2YyTKETuAHpm2dNDpAZwDDsx7p9qmRBCGVcQwMTpx78Ce1lBzAd56PkO67gww8uxEmnU46nIoCqPUcgMT0KxqcBktKkoyH6RlVGEhbkgJwgfQ0PaglWBT189QlAweCn+BrG6n7W3kG09mW14kNAkOQvFclSQAnLQWDXZib+hh9YCF4AgciTx3IkCOJ7pKFmpSBA+XAOSUqXeh9hru9vjdKDfuRgyEEiUaF/j/yypuqK+iobxCmZoftxPrpq1HXwqLZVeNL5xjiOE4YsByg3+Ll6gJarh4sCj+GmvDKuMgrY2uwH0VJQhEWgiip1Kilg8nol8ikQW31ZFQMKDpQg1pwRdP5Bo4hDAgHrVFaOV3WiM+jcYUtYQw0uiB/gMveFfGetlAdyFbOWV3mS+1OGhWj73Lz5ITq5GnRSUTlUAcVA81XIVZUNaI8RG5+fcj3py6cjddsW4hm76KFuqKFRAg4DIdSG9AeApZ2LlqwwCdaqLk2F1nhShkZKlUdzGut0UXHZq10f6O57NnH+6xQWW84FzhcOBccWKwxZK9jucnJSNjAJVosqJCOQXF5nV75Wfl4nAtVTrHpY/Ava9JLQLaxO1fGSg1gfHMu5RzutbK3FJLmgqNubv892eymrn1DR8oSdwwkC4acBo/s6TlR1hKfOwbuzmbuI/hsyTajlvlkFSECa+RZtxa298iznFAaU+iwQ1YaIqW9kYmca0Wsaq8g2/DnEtlzxm6nA7PIGDmQId0yhcIclODcKpqqlQV26n02G2SRZDgQUUphjPX3WkRCpqxuDwfiqrcmNO8NlhQ2PfUGTVztE4vBgXhAW0ZAS3OIqu3BqmYqyZFaayWBKKSzi0gZMVVBCFGShLqzOZD7AlD1gZRRXaMDqKrqVAONTaYzGj2e7SJWBdZ2cZJY0s3kb3ZSiQfqQCUCL8Epn/jYOAoslSrhlXfWL1Wlj+w0FOHe2hosqigYC/1tVxd1pAetYjgleswEK7pznypGbaQMRuAkCTf5uEEkZMOJA+2XdmT7Qa0YL6w5FTW0GMZGK5YDp6LbKwOpwlksG2xo0BalJpxwTzV/0B2wKeUucLDLyG7y4yolevzEwaVAtCA24nGg5O1d631OL5v8ZoA0gjNKZZRCHbxe5uCFw86cFi/NMdBeiZH2xgkV3XQITAFGdZzLDiRg6gEFgbCg/aRAYi6ccrI5YFVTS58d8Nmdg9S9FBwa6Xlj2n0G566xmIBzSkmvtXJOz8ncjpqHh2+A3/nwIfzjkLFAy9LdZoUrVmSaC/52TeM9v7jNFiZluwHIEJIx3rTCREzrnGouMoRaduewe9DGpaQkqulO3TBJhdtyvdH7SbcLnK3KDSnNYloOBhQpcOC7Y4kRGF03744xgxhwgbaz9ZGsAd5dbWecIby1DPtGWuz8WlKequSjUJwE0hDMSVSkOTBdIHRYOQvFv/bmAGO66lnfJS8VrVHbk23TxtELjQClKTBhIYIC2+aG39WIR1fH7434vgVSuz4Rqf3GOUseqFyngqcZ3rtBCGuEJRe8gW2kU3MHuoZfeB3vpzZtDwHcnQmFyKdK5GBx/PSK0/l4p/OsXTlWAjgYCUhCOwii73k23Ph9PkknHO/icn2vFh8rXOqMzpk1Kxrx/1qDhbjerCVOBG5AQnSolX+s0JjIBYBGr5UoyrhIX9l/xQn3vlCHJ1o0cT3ogipXWP3TGjVAFNo6pULslY6cfezCj4/p7O/fNsulweVsr0CTi/MNrJp7dPNbG9qSTDnRWOkoRLlgtArIRlXgAqsMdeTZFV/Nejek+HJpU3Z2iKyRYJ4x7KyTnnPDOTM1VwEGDRg0rrh45ejAxElmTchc3ZUGlJgOH16ofqbFXRByjr02lBnh0GDRSCmnd1eQjBQB1WijnRxq0d4xQ3cYo6b7nl9Pxizd7CNutnXXLBlsehDed0e7EF9tYxBuHjMGmXnjgi2CLZn8Yvg6dcHpiLd/h11EjwCRdJTUQNYnfikIFtAQe5JMoUgxtS+9FGa3YK3GdDHB+pTu7nBs1EtO7b9rIrnMC5KlSTQ4p4Uen5mgJog9MaWUlktRYWCPbuncG7pzTWMONNRzOlwrTeNZQiWSbF6FtuisIl8xgCIlg48FBLtZQFVsTyZIeQvWUtVgrSOYP2cEeauAQaoEDBIDSkvkLErfeZt5MhqAuJ0lTC4YhH2Cwrbj29Nq2XNPZSsEg7DUV9066TyLKeyUIEJkAmgVkhxTSF14jdWvPD6zEozclrtRxeBHH0E6N/kPoS6jI9gBhRpfQoJpNRo5xchmk5UcQiRylOvVVqPqJBYY/A8TdZxFyLSluAELC8VjYeEXpJKsCg+agEgeGYCsYZkHR1nm8Rri3HAdOFm04LllwUc58gGUBQkIrUlZM9numlj1ILZ+GFPpFbWULV/rxgKTkSqTKLDdepQdCN4i/VH5ll6Pj5l22mDQmYzzWeQ4quaOTHX93LPiceImsILNm+ZYq/YqPBzZMfRi7k7yiSDxl/zehJdt7IqFkk6HYBMVw4o1G0vjSV98wcTlSh/FZNFRug1RI2cwrcXyEAxS625oF/eSkm5cnWw9S2DIW+JgSN2KEL/AsI3aPKZ5UG5O/9W1ZHGyPiCKsxb9qkNvPVeDrrY5V3LEXYoB0ZG0jgJVkmJAdd14sWSgWvDEi9GKUs71GFp7iF0P5c+ZjuVkmmE8eM5Q8P11VsVKeR7x5oI8VII+usOlzKEuaLx4oi1/esN7bxAPoRVIpgZRS1QEPtgkUlPwhjLMckvKol5c9hBhZO7Ss7vgtYjjuKF+hyn6OSniWZK1ksJi18SqCRzo0L7hDwj7GB5PeUpbQtcurWZ09TnT1B8bRSMJXpDa6YK9pRsMCqelUhT4TxHePGKjtWWq0AYKt5Zq2YwMF9TfB68sdx9AyaCVy8VetvpDV3BczdgXCimlcuDHTYhs2VHy+UdiRY5ONQ+xqqT1hmt3cSfd3PKYWMg+F1ZIAZoC7ESRnSRS857uAhl14LKrq1tcHtg1y3N3Czk7dZiqBfOJ3bQvT/FEtoOHTqVTJtj3R3QU2Iqoc1nPTuNSLk4v2QFhrdBuKMkBO2sElaThbXrQ4+lzYjxqvpzq2IisD3RmBVa5/VRMfSe1y45bdpBXZg8cjgvtsWXaObQIEiw5zwuIeZSh6SOTdmscTczL2aAEuB6lN44R3KiLg+5YhQLHKnSsnAYQVGhKfRDP/wPdIMkGpcsBAA==")))
print("Cached profile rows:", len(CHECKPOINT.read_text().splitlines()))


## 3. Load the open-source LLM

In [ ]:
CHECKPOINT_READY = Path("fixed_tolerance_experiment_results/profile_extractions.jsonl").exists()
if CHECKPOINT_READY:
    tokenizer = model = None
    MODEL_REVISION = "cached-profile-checkpoint"
    print("Using cached LLM profiles; GPU/model loading skipped.")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    assert torch.cuda.is_available(), "Select a GPU runtime in Colab."
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization_config,
        torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
    ).eval()
    MODEL_REVISION = getattr(model.config, "_commit_hash", None)
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
if CHECKPOINT_READY:
    print("Smoke test skipped: using the validated profile checkpoint.")
else:
    smoke_student_id = int(reports_df.student_id.iloc[0])
    smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
    print(smoke_raw)
    print(smoke_profile)


## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint uses the same filename as the earlier notebooks. If it is present in the Colab workspace, completed LLM profiles are reused. The allocation experiment below is CPU-only.


## 8. Train/validation/test design, weak tiers, and Pareto completion


In [ ]:
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 64
N_VALIDATION_STUDENTS = 32
N_TEST_STUDENTS = 32
N_VALIDATION_ORDERS = 20
N_TEST_ORDERS = 20
MARKET_CAPACITY_PER_COURSE = 5
LOGIT_C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

# Thresholds are measured in within-student standardized predicted-score units.
# tau=0 is the strict-proxy baseline.
TAU_GRID = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]
ROBUST_CVAR_QUANTILE = 0.90
ROBUST_CVAR_WEIGHT = 0.50

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def strict_pair_target(rows):
    """Strict ground truth from raw utility; bundle tuple breaks exact equality."""
    targets = []
    for row in rows.itertuples(index=False):
        utility_a = float(row.true_utility_a)
        utility_b = float(row.true_utility_b)
        bundle_a = tuple(sorted(map(int, row.bundle_a)))
        bundle_b = tuple(sorted(map(int, row.bundle_b)))
        targets.append(int(
            utility_a > utility_b
            or (utility_a == utility_b and bundle_a < bundle_b)
        ))
    return np.asarray(targets, dtype=int)


def fit_strict_pairwise_model(rows, return_cv=False):
    """Fit a coherent linear bundle score using strict pair directions only."""
    X_local = rows[FEATURE_NAMES].to_numpy()
    y_local = strict_pair_target(rows)
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    c_rows = []
    for C in LOGIT_C_GRID:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X_local, y_local, groups):
            model = LogisticRegression(
                C=C, fit_intercept=False, solver="liblinear",
                random_state=SEED, max_iter=2_000,
            )
            model.fit(X_local[fit_idx], y_local[fit_idx])
            oof[hold_idx] = model.predict_proba(X_local[hold_idx])[:, 1]
        predicted = (oof >= 0.5).astype(int)
        c_rows.append({
            "C": C,
            "oof_direction_accuracy": accuracy_score(y_local, predicted),
            "oof_log_loss": float(-np.mean(
                y_local * np.log(np.clip(oof, 1e-12, 1))
                + (1 - y_local) * np.log(np.clip(1 - oof, 1e-12, 1))
            )),
        })
    cv_results = pd.DataFrame(c_rows).sort_values(
        ["oof_direction_accuracy", "oof_log_loss", "C"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    best_C = float(cv_results.loc[0, "C"])
    model = LogisticRegression(
        C=best_C, fit_intercept=False, solver="liblinear",
        random_state=SEED, max_iter=2_000,
    ).fit(X_local, y_local)
    return (model, best_C, cv_results) if return_cv else model


def heldout_direction_metrics(rows, model):
    y_local = strict_pair_target(rows)
    probabilities = model.predict_proba(rows[FEATURE_NAMES].to_numpy())[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "direction_accuracy": float(accuracy_score(y_local, predictions)),
        "direction_log_loss": float(-np.mean(
            y_local * np.log(np.clip(probabilities, 1e-12, 1))
            + (1 - y_local) * np.log(np.clip(1 - probabilities, 1e-12, 1))
        )),
        "n_pairwise_queries": int(len(rows)),
    }


def build_candidate_market(rows, model):
    """Build standardized proxy scores and hidden true values on one market."""
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        raw = {
            b: float(np.dot(model.coef_[0], bundle_features(profiles[i], b)))
            for b in bundles
        }
        raw_values = np.array(list(raw.values()), dtype=float)
        scale = float(np.std(raw_values))
        if scale < 1e-8:
            scale = 1.0
        center = float(np.mean(raw_values))
        predicted[i] = {b: (value - center) / scale for b, value in raw.items()}

        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(values)) for b, values in observations.items()}

        # Outside option is always a separate, strictly worst tier.
        predicted[i][tuple()] = min(predicted[i].values()) - 10.0
        truth[i][tuple()] = min(truth[i].values()) - 2.0
    return predicted, truth


def strict_ranks(score_map):
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    return {bundle: rank for rank, bundle in enumerate(ordered)}


def rank_maps(score_maps):
    return {i: strict_ranks(scores) for i, scores in score_maps.items()}


def tier_map(score_map, tau):
    """Create transitive score-anchored tiers; the outside option stays worst."""
    outside = tuple()
    ordered = sorted((b for b in score_map if b), key=lambda b: (-score_map[b], b))
    tiers = {}
    if ordered:
        current_tier = 0
        anchor = score_map[ordered[0]]
        tiers[ordered[0]] = current_tier
        for bundle in ordered[1:]:
            if anchor - score_map[bundle] > tau + 1e-12:
                current_tier += 1
                anchor = score_map[bundle]
            tiers[bundle] = current_tier
        tiers[outside] = current_tier + 1
    else:
        tiers[outside] = 0
    return tiers


def tier_maps(score_maps, tau):
    return {i: tier_map(scores, tau) for i, scores in score_maps.items()}


def predicted_tie_rate(tiers):
    tied = total = 0
    for i in tiers:
        bundles = [b for b in tiers[i] if b]
        for a, b in combinations(bundles, 2):
            total += 1
            tied += int(tiers[i][a] == tiers[i][b])
    return float(tied / total) if total else 0.0


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def run_weak_rsd(decision_tiers, capacities, seed):
    """Weak RSD with a fixed report-independent lexicographic rule within a tier."""
    rng = np.random.default_rng(seed)
    order = list(map(int, rng.permutation(sorted(decision_tiers))))
    priority_position = {i: position for position, i in enumerate(order)}
    remaining = capacities.copy()
    allocation = {}
    for i in order:
        feasible = [b for b in decision_tiers[i] if fits(b, remaining)]
        best_tier = min(decision_tiers[i][b] for b in feasible)
        chosen = min(b for b in feasible if decision_tiers[i][b] == best_tier)
        allocation[i] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, priority_position


def proxy_pareto_completion(initial_allocation, proxy_tiers, capacities):
    """Proxy-Pareto completion: no proxy loss, maximum total tier improvement."""
    students = sorted(proxy_tiers)
    allowed = {
        i: [
            b for b in proxy_tiers[i]
            if proxy_tiers[i][b] <= proxy_tiers[i][initial_allocation[i]]
        ]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []

    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))

    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value

    tier_gain = np.array([
        proxy_tiers[i][initial_allocation[i]] - proxy_tiers[i][b]
        for i, b in variables
    ], dtype=float)
    # The tiny deterministic term resolves equal-gain optima without changing
    # the integer-valued primary objective.
    deterministic = np.array([
        (student_pos + 1) * 1e-7 + (bundle_pos + 1) * 1e-10
        for student_pos, i in enumerate(students)
        for bundle_pos, b in enumerate(allowed[i])
    ])
    objective = -tier_gain + deterministic
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        raise RuntimeError(f"Pareto-completion MILP failed: {result.message}")
    completed = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    assert len(completed) == len(students)
    assert all(
        proxy_tiers[i][completed[i]] <= proxy_tiers[i][initial_allocation[i]]
        for i in students
    )
    return completed, int(round(tier_gain @ (result.x > 0.5))), result.message


def run_weak_rsd_with_completion(proxy_tiers, capacities, seed):
    initial, priority = run_weak_rsd(proxy_tiers, capacities, seed)
    completed, total_tier_gain, status = proxy_pareto_completion(
        initial, proxy_tiers, capacities
    )
    return completed, priority, initial, total_tier_gain, status


def run_strict_rsd(decision_ranks, capacities, seed):
    tiers = {i: dict(ranks) for i, ranks in decision_ranks.items()}
    return run_weak_rsd(tiers, capacities, seed)


def max_pareto_improvements(allocation, true_ranks, capacities):
    """Maximum strictly improved students subject to nobody becoming worse."""
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lower, upper = [], [], []
    for i in students:
        rows.append({index[(i, b)]: 1.0 for b in allowed[i]})
        lower.append(1.0); upper.append(1.0)
    for course in COURSE_IDS:
        rows.append({index[(i, b)]: 1.0 for i, b in variables if course in b})
        lower.append(-np.inf); upper.append(float(capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value
    objective = np.array([
        -float(true_ranks[i][b] < true_ranks[i][allocation[i]])
        for i, b in variables
    ])
    result = milp(
        c=objective,
        integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return np.nan, result.message
    chosen = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    count = sum(true_ranks[i][chosen[i]] < true_ranks[i][allocation[i]] for i in students)
    return int(count), result.message


def priority_blocking_metrics(allocation, true_ranks, capacities, priority_position):
    """Count strictly true-preferred bundles obtainable from empty/lower-priority seats."""
    holders = {c: [] for c in COURSE_IDS}
    for j, bundle in allocation.items():
        for c in bundle:
            holders[c].append(j)
    blocking = []
    for i in sorted(allocation):
        own = allocation[i]
        own_rank = true_ranks[i][own]
        for bundle, rank in true_ranks[i].items():
            if rank >= own_rank:
                continue
            obtainable = True
            for c in bundle:
                if c in own:
                    continue
                course_holders = holders[c]
                if len(course_holders) < capacities[c]:
                    continue
                if any(priority_position[i] < priority_position[j] for j in course_holders):
                    continue
                obtainable = False
                break
            if obtainable:
                blocking.append((i, bundle))
    denominator = sum(max(len(true_ranks[i]) - 1, 1) for i in true_ranks)
    return {
        "priority_blocking_bundles": len(blocking),
        "normalized_blocking_rate": float(len(blocking) / denominator),
        "priority_blocking_students": len({i for i, _ in blocking}),
        "zero_priority_blocking": int(len(blocking) == 0),
    }


def evaluate_allocation(allocation, true_ranks, capacities, priority_position):
    pe_count, solver_status = max_pareto_improvements(allocation, true_ranks, capacities)
    assigned_ranks = np.array([true_ranks[i][allocation[i]] for i in sorted(allocation)])
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_violation_rate": float(pe_count / len(allocation)) if np.isfinite(pe_count) else np.nan,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
        "mean_true_rank": float(assigned_ranks.mean()),
        "worst_true_rank": int(assigned_ranks.max()),
        "pe_solver_status": solver_status,
    }
    metrics.update(priority_blocking_metrics(
        allocation, true_ranks, capacities, priority_position
    ))
    return metrics


def compare_to_oracle(allocation, oracle_allocation, true_ranks):
    students = sorted(true_ranks)
    losses = np.array([
        true_ranks[i][allocation[i]] - true_ranks[i][oracle_allocation[i]]
        for i in students
    ])
    return {
        "allocation_agreement_rate": float(np.mean([
            allocation[i] == oracle_allocation[i] for i in students
        ])),
        "mean_true_rank_loss_vs_oracle": float(losses.mean()),
        "median_true_rank_loss_vs_oracle": float(np.median(losses)),
        "share_worse_than_oracle": float(np.mean(losses > 0)),
        "share_better_than_oracle": float(np.mean(losses < 0)),
        "worst_student_rank_loss": int(losses.max()),
    }


def cvar(values, quantile=ROBUST_CVAR_QUANTILE):
    values = np.asarray(values, dtype=float)
    cutoff = np.quantile(values, quantile)
    tail = values[values >= cutoff - 1e-12]
    return float(tail.mean())


def robust_risk(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean() + ROBUST_CVAR_WEIGHT * cvar(values))


def positive_true_values(true_scores, epsilon=1e-4):
    """CEEI-compatible within-student candidate-menu min-max normalization."""
    result = {}
    for i, score_map in true_scores.items():
        nonempty = [float(v) for b, v in score_map.items() if b]
        low, high = min(nonempty), max(nonempty)
        if high - low <= 1e-12:
            values = {b: 1.0 for b in score_map if b}
        else:
            values = {
                b: float(epsilon + (1.0 - epsilon) * (float(v) - low) / (high - low))
                for b, v in score_map.items() if b
            }
        values[tuple()] = float(epsilon)
        result[i] = values
    return result


def nash_welfare_metrics(allocation, true_scores):
    normalized = positive_true_values(true_scores)
    assigned = np.array([normalized[i][allocation[i]] for i in sorted(allocation)], dtype=float)
    log_nw = float(np.mean(np.log(assigned)))
    return {
        "true_log_nash_welfare": log_nw,
        "true_nash_welfare": float(np.exp(log_nw)),
        "mean_normalized_true_utility": float(assigned.mean()),
        "minimum_normalized_true_utility": float(assigned.min()),
        "p10_normalized_true_utility": float(np.quantile(assigned, 0.10)),
    }, normalized


## 9. Sanity checks for weak tiers and Pareto completion


In [ ]:
# Tier construction must be transitive and the outside option strictly worst.
toy_scores = {(1,): 1.00, (2,): 0.96, (3,): 0.91, tuple(): -10.0}
toy_tiers = tier_map(toy_scores, tau=0.05)
assert toy_tiers[(1,)] == toy_tiers[(2,)]
assert toy_tiers[(3,)] > toy_tiers[(2,)]
assert toy_tiers[tuple()] > max(toy_tiers[b] for b in toy_tiers if b)

# In a strict proxy order, serial dictatorship is already proxy-PE, so completion
# must have zero total strict-tier gain (it may only select an equal objective optimum).
toy_course_1, toy_course_2 = COURSE_IDS[:2]
toy_proxy_tiers = {
    0: {(toy_course_1,): 0, (toy_course_2,): 1, tuple(): 2},
    1: {(toy_course_1,): 0, (toy_course_2,): 1, tuple(): 2},
    2: {(toy_course_2,): 0, (toy_course_1,): 1, tuple(): 2},
}
toy_capacities = {c: 1 for c in COURSE_IDS}
toy_allocation, toy_priority = run_weak_rsd(toy_proxy_tiers, toy_capacities, SEED)
toy_completed, toy_gain, _ = proxy_pareto_completion(
    toy_allocation, toy_proxy_tiers, toy_capacities
)
assert toy_gain == 0
assert all(
    toy_proxy_tiers[i][toy_completed[i]] <= toy_proxy_tiers[i][toy_allocation[i]]
    for i in toy_proxy_tiers
)
print("Weak-tier and Pareto-completion sanity checks passed.")


## 10. Repeated train/validation/locked-test experiment


In [ ]:
all_ids = np.array(sorted(queries.student_id.unique()))
assert len(all_ids) == N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS + N_TEST_STUDENTS
split_rng = np.random.default_rng(SEED + 2_400)

manifest_rows = []
cv_frames = []
heldout_metric_rows = []
validation_rows = []
selection_rows = []
test_run_rows = []
comparison_rows = []
assigned_rows = []

for split_id in tqdm(range(N_REPEATED_SPLITS), desc="Weak-RSD repeated splits"):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    validation_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS]))
    test_ids = set(map(int, shuffled[-N_TEST_STUDENTS:]))
    assert not (train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids)

    for role, ids in [
        ("train", train_ids), ("validation", validation_ids), ("test", test_ids)
    ]:
        manifest_rows.extend(
            {"split_id": split_id, "student_id": i, "role": role} for i in ids
        )

    train = queries[queries.student_id.isin(train_ids)].copy()
    validation = queries[queries.student_id.isin(validation_ids)].copy()
    test = queries[queries.student_id.isin(test_ids)].copy()
    model, best_C, cv_results = fit_strict_pairwise_model(train, return_cv=True)
    cv_results["split_id"] = split_id
    cv_frames.append(cv_results)

    for role, rows in [("validation", validation), ("test", test)]:
        heldout_metric_rows.append({
            "split_id": split_id, "role": role,
            **heldout_direction_metrics(rows, model),
        })

    val_predicted_scores, val_true_scores = build_candidate_market(validation, model)
    val_true_ranks = rank_maps(val_true_scores)
    val_tau_tiers = {tau: tier_maps(val_predicted_scores, tau) for tau in TAU_GRID}

    for tau in TAU_GRID:
        tie_rate = predicted_tie_rate(val_tau_tiers[tau])
        for order_run in range(N_VALIDATION_ORDERS):
            order_seed = SEED + 20_000_000 + 100_000 * split_id + order_run
            allocation, priority, initial, tier_gain, completion_status = run_weak_rsd_with_completion(
                val_tau_tiers[tau], market_capacities, order_seed
            )
            metrics = evaluate_allocation(
                allocation, val_true_ranks, market_capacities, priority
            )
            validation_rows.append({
                "split_id": split_id, "tau": tau, "order_run": order_run,
                "predicted_tie_rate": tie_rate,
                "completion_total_tier_gain": tier_gain,
                "completion_changed_share": float(np.mean([
                    allocation[i] != initial[i] for i in allocation
                ])),
                "completion_solver_status": completion_status,
                **metrics,
            })

    current_validation = pd.DataFrame(
        [row for row in validation_rows if row["split_id"] == split_id]
    )
    threshold_table = []
    for tau, group in current_validation.groupby("tau"):
        threshold_table.append({
            "tau": float(tau),
            "pe_mean": float(group.pareto_violation_rate.mean()),
            "pe_cvar90": cvar(group.pareto_violation_rate),
            "pe_robust_risk": robust_risk(group.pareto_violation_rate),
            "blocking_mean": float(group.normalized_blocking_rate.mean()),
            "blocking_cvar90": cvar(group.normalized_blocking_rate),
            "blocking_robust_risk": robust_risk(group.normalized_blocking_rate),
            "predicted_tie_rate": float(group.predicted_tie_rate.iloc[0]),
        })
    threshold_table = pd.DataFrame(threshold_table)
    pe_tau = float(threshold_table.sort_values(
        ["pe_robust_risk", "blocking_robust_risk", "tau"]
    ).iloc[0].tau)
    block_tau = float(threshold_table.sort_values(
        ["blocking_robust_risk", "pe_robust_risk", "tau"]
    ).iloc[0].tau)
    for row in threshold_table.to_dict("records"):
        selection_rows.append({
            "split_id": split_id,
            "selected_for_pe": int(row["tau"] == pe_tau),
            "selected_for_blocking": int(row["tau"] == block_tau),
            **row,
        })

    test_predicted_scores, test_true_scores = build_candidate_market(test, model)
    test_predicted_ranks = rank_maps(test_predicted_scores)
    test_true_ranks = rank_maps(test_true_scores)
    selected_tiers = {
        "llm_proxy_weak_pe_calibrated": tier_maps(test_predicted_scores, pe_tau),
        "llm_proxy_weak_block_calibrated": tier_maps(test_predicted_scores, block_tau),
    }

    for order_run in range(N_TEST_ORDERS):
        order_seed = SEED + 30_000_000 + 100_000 * split_id + order_run
        oracle_allocation, oracle_priority = run_strict_rsd(
            test_true_ranks, market_capacities, order_seed
        )
        strict_allocation, strict_priority = run_strict_rsd(
            test_predicted_ranks, market_capacities, order_seed
        )
        assert oracle_priority == strict_priority

        method_allocations = {
            "oracle_strict_rsd": (oracle_allocation, oracle_priority, 0.0, 0, 0.0),
            "llm_proxy_strict_rsd": (strict_allocation, strict_priority, 0.0, 0, 0.0),
        }
        for method, tiers in selected_tiers.items():
            completed, priority, initial, tier_gain, _ = run_weak_rsd_with_completion(
                tiers, market_capacities, order_seed
            )
            assert priority == oracle_priority
            tau = pe_tau if method.endswith("pe_calibrated") else block_tau
            method_allocations[method] = (
                completed, priority, tau, tier_gain,
                float(np.mean([completed[i] != initial[i] for i in completed])),
            )

        oracle_nw, normalized_truth = nash_welfare_metrics(oracle_allocation, test_true_scores)
        for method, (allocation, priority, tau, tier_gain, changed_share) in method_allocations.items():
            metrics = evaluate_allocation(
                allocation, test_true_ranks, market_capacities, priority
            )
            nw_metrics, _ = nash_welfare_metrics(allocation, test_true_scores)
            metrics.update(nw_metrics)
            for i in sorted(allocation):
                assigned_rows.append({
                    "split_id": split_id, "order_run": order_run,
                    "method": method, "student_id": i,
                    "assigned_bundle": repr(allocation[i]),
                    "raw_true_utility": test_true_scores[i][allocation[i]],
                    "normalized_true_utility": normalized_truth[i][allocation[i]],
                })
            tiers_for_tie_rate = (
                selected_tiers[method] if method in selected_tiers else None
            )
            test_run_rows.append({
                "split_id": split_id, "order_run": order_run,
                "method": method, "best_logit_C": best_C,
                "selected_tau": tau,
                "predicted_tie_rate": predicted_tie_rate(tiers_for_tie_rate) if tiers_for_tie_rate else 0.0,
                "completion_total_tier_gain": tier_gain,
                "completion_changed_share": changed_share,
                **metrics,
            })
            if method != "oracle_strict_rsd":
                comparison_rows.append({
                    "split_id": split_id, "order_run": order_run,
                    "method": method,
                    **compare_to_oracle(allocation, oracle_allocation, test_true_ranks),
                    "nash_welfare_ratio": float(np.exp(
                        nw_metrics["true_log_nash_welfare"] - oracle_nw["true_log_nash_welfare"]
                    )),
                    "log_nash_welfare_gap": float(
                        nw_metrics["true_log_nash_welfare"] - oracle_nw["true_log_nash_welfare"]
                    ),
                })

split_manifest = pd.DataFrame(manifest_rows)
cv_results_all = pd.concat(cv_frames, ignore_index=True)
heldout_metrics = pd.DataFrame(heldout_metric_rows)
validation_results = pd.DataFrame(validation_rows)
threshold_selection = pd.DataFrame(selection_rows)
test_results = pd.DataFrame(test_run_rows)
paired_comparisons = pd.DataFrame(comparison_rows)

summary = (
    test_results.groupby("method", as_index=False)
    .agg(
        pe_rate=("pareto_efficient", "mean"),
        mean_pareto_improvable_students=("pareto_improvable_students", "mean"),
        mean_pe_score=("pe_score", "mean"),
        zero_block_rate=("zero_priority_blocking", "mean"),
        mean_blocking_bundles=("priority_blocking_bundles", "mean"),
        median_blocking_bundles=("priority_blocking_bundles", "median"),
        p90_blocking_bundles=("priority_blocking_bundles", lambda x: float(np.quantile(x, 0.90))),
        worst_blocking_bundles=("priority_blocking_bundles", "max"),
        mean_blocking_students=("priority_blocking_students", "mean"),
        worst_blocking_students=("priority_blocking_students", "max"),
        mean_true_rank=("mean_true_rank", "mean"),
        mean_selected_tau=("selected_tau", "mean"),
        mean_predicted_tie_rate=("predicted_tie_rate", "mean"),
        mean_completion_changed_share=("completion_changed_share", "mean"),
        mean_true_nash_welfare=("true_nash_welfare", "mean"),
        mean_minimum_utility=("minimum_normalized_true_utility", "mean"),
    )
)
comparison_summary = (
    paired_comparisons.groupby("method")
    .agg(
        allocation_agreement_rate=("allocation_agreement_rate", "mean"),
        mean_true_rank_loss_vs_oracle=("mean_true_rank_loss_vs_oracle", "mean"),
        share_worse_than_oracle=("share_worse_than_oracle", "mean"),
        share_better_than_oracle=("share_better_than_oracle", "mean"),
        worst_student_rank_loss=("worst_student_rank_loss", "max"),
        mean_nash_welfare_ratio=("nash_welfare_ratio", "mean"),
        median_nash_welfare_ratio=("nash_welfare_ratio", "median"),
        worst_nash_welfare_ratio=("nash_welfare_ratio", "min"),
    ).reset_index()
)

display(summary)
display(comparison_summary)
display(heldout_metrics.groupby("role").agg({
    "direction_accuracy": ["mean", "std", "min"],
    "direction_log_loss": ["mean", "std"],
}))

oracle_rows = test_results[test_results.method.eq("oracle_strict_rsd")]
oracle_pe_failures = int((oracle_rows.pareto_efficient != 1).sum())
oracle_block_failures = int((oracle_rows.zero_priority_blocking != 1).sum())
print("Oracle PE failure runs:", oracle_pe_failures)
print("Oracle priority-blocking failure runs:", oracle_block_failures)
assert oracle_pe_failures == 0
assert oracle_block_failures == 0


## 11. Visualize validation paths and locked-test performance


In [ ]:
FIGURE_DIR = Path("weak_rsd_pareto_completion_figures")
FIGURE_DIR.mkdir(exist_ok=True)

validation_curve = (
    validation_results.groupby("tau", as_index=False)
    .agg(
        tie_rate=("predicted_tie_rate", "mean"),
        pareto_violation_rate=("pareto_violation_rate", "mean"),
        normalized_blocking_rate=("normalized_blocking_rate", "mean"),
    )
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(validation_curve.tau, validation_curve.pareto_violation_rate, marker="o", color="#dc2626")
axes[0].set(title="Validation Pareto violations", xlabel="Indifference threshold", ylabel="Mean violation rate")
axes[1].plot(validation_curve.tau, validation_curve.normalized_blocking_rate, marker="o", color="#2563eb")
axes[1].set(title="Validation priority blocking", xlabel="Indifference threshold", ylabel="Mean normalized blocking rate")
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "validation_threshold_paths.png", dpi=180)
plt.show()

method_order = [
    "oracle_strict_rsd", "llm_proxy_strict_rsd",
    "llm_proxy_weak_pe_calibrated", "llm_proxy_weak_block_calibrated",
]
plot_summary = summary.set_index("method").reindex(method_order).reset_index()
short_labels = ["Oracle strict", "Proxy strict", "Weak: PE", "Weak: block"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(short_labels, plot_summary.mean_pareto_improvable_students, color=["#64748b", "#dc2626", "#7c3aed", "#2563eb"])
axes[0].set(title="Locked-test Pareto violations", ylabel="Mean Pareto-improvable students")
axes[1].bar(short_labels, plot_summary.mean_blocking_bundles, color=["#64748b", "#dc2626", "#7c3aed", "#2563eb"])
axes[1].set(title="Locked-test priority blocking", ylabel="Mean blocking bundles")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "locked_test_method_comparison.png", dpi=180)
plt.show()

selected = threshold_selection[
    threshold_selection.selected_for_pe.eq(1) | threshold_selection.selected_for_blocking.eq(1)
]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, column, title, color in [
    (axes[0], "selected_for_pe", "PE-calibrated thresholds", "#7c3aed"),
    (axes[1], "selected_for_blocking", "Blocking-calibrated thresholds", "#2563eb"),
]:
    values = threshold_selection.loc[threshold_selection[column].eq(1), "tau"]
    counts = values.value_counts().reindex(TAU_GRID, fill_value=0)
    ax.bar([str(x) for x in counts.index], counts.values, color=color)
    ax.set(title=title, xlabel="Selected threshold", ylabel="Number of splits")
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "selected_threshold_distribution.png", dpi=180)
plt.show()

proxy_test = test_results[~test_results.method.eq("oracle_strict_rsd")]
split_robustness = (
    proxy_test.groupby(["method", "split_id"], as_index=False)
    .agg(
        mean_pareto_improvable=("pareto_improvable_students", "mean"),
        worst_pareto_improvable=("pareto_improvable_students", "max"),
        mean_blocks=("priority_blocking_bundles", "mean"),
        worst_blocks=("priority_blocking_bundles", "max"),
        pe_rate=("pareto_efficient", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
nw_plot = summary.set_index("method").reindex(method_order)
axes[0].bar(short_labels, nw_plot.mean_true_nash_welfare, color=["#64748b", "#dc2626", "#7c3aed", "#2563eb"])
axes[0].set(title="Locked-test true Nash welfare", ylabel="Geometric mean normalized utility")
for method, label, color in zip(method_order[1:], short_labels[1:], ["#dc2626", "#7c3aed", "#2563eb"]):
    values = paired_comparisons.loc[paired_comparisons.method.eq(method), "nash_welfare_ratio"]
    axes[1].hist(values, bins=18, alpha=0.45, label=label, color=color)
axes[1].axvline(1.0, color="black", linestyle="--")
axes[1].set(title="Proxy/oracle Nash-welfare ratios", xlabel="Nash-welfare ratio", ylabel="Paired runs")
axes[1].legend()
for ax in axes: ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "weak_rsd_nash_welfare.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("weak_rsd_pareto_completion_results")
OUTPUT_DIR.mkdir(exist_ok=True)
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
cv_results_all.to_csv(OUTPUT_DIR / "pairwise_model_cv.csv", index=False)
heldout_metrics.to_csv(OUTPUT_DIR / "heldout_pairwise_metrics.csv", index=False)
validation_results.to_csv(OUTPUT_DIR / "validation_threshold_runs.csv", index=False)
threshold_selection.to_csv(OUTPUT_DIR / "threshold_selection.csv", index=False)
test_results.to_csv(OUTPUT_DIR / "locked_test_runs.csv", index=False)
pd.DataFrame(assigned_rows).to_csv(OUTPUT_DIR / "assigned_student_utilities.csv", index=False)
paired_comparisons.to_csv(OUTPUT_DIR / "proxy_oracle_paired_comparisons.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
comparison_summary.to_csv(OUTPUT_DIR / "comparison_summary.csv", index=False)
split_robustness.to_csv(OUTPUT_DIR / "split_robustness.csv", index=False)
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "mechanism": "weak proxy RSD followed by proxy-Pareto completion",
    "preference_model_training_information": "strict pairwise A/B directions from training students only",
    "threshold_selection_information": "validation true PE violations or true normalized priority-blocking bundles",
    "excluded_from_preference_model_training": [
        "utility magnitude", "indifference labels", "allocation utility",
        "Pareto violations", "blocking counts",
    ],
    "domain": "observed candidate bundles plus outside option",
    "nash_welfare": "locked-test geometric mean of within-student min-max normalized true assigned utilities",
    "utility_normalization": "candidate-menu min-max to [1e-4,1], outside fixed at 1e-4",
    "nw_not_used_for_threshold_selection": True,
    "scope_warning": "PE and blocking claims are relative to this candidate-bundle domain.",
    "n_repeated_splits": N_REPEATED_SPLITS,
    "split_sizes": {
        "train": N_TRAIN_STUDENTS,
        "validation": N_VALIDATION_STUDENTS,
        "locked_test": N_TEST_STUDENTS,
    },
    "validation_orders_per_market": N_VALIDATION_ORDERS,
    "test_orders_per_market": N_TEST_ORDERS,
    "capacity_per_course": MARKET_CAPACITY_PER_COURSE,
    "threshold_grid": TAU_GRID,
    "threshold_units": "within-student standard deviations of predicted bundle score",
    "robust_selection": {
        "criterion": "mean loss + weight * CVaR90 loss",
        "cvar_quantile": ROBUST_CVAR_QUANTILE,
        "cvar_weight": ROBUST_CVAR_WEIGHT,
    },
    "oracle_pe_failures": oracle_pe_failures,
    "oracle_priority_blocking_failures": oracle_block_failures,
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
